# Семинар 1. Gymnasium, многорукие бандиты, знакомство с MDP-средой

План:

1. Интерфейс Gymnasium на примере `FrozenLake-v1`
2. Своя среда многорукого бандита
3. Агенты: ε-greedy, UCB1, Thompson Sampling
4. Сравнение regret


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

rng = np.random.default_rng(42)


ModuleNotFoundError: No module named 'gymnasium'

## 1. Интерфейс Gymnasium

Любая среда в Gymnasium следует единому интерфейсу:

* `env.reset(seed=...)` -> `(observation, info)` — сбросить среду в начальное состояние
* `env.step(action)` -> `(observation, reward, terminated, truncated, info)` — сделать шаг
* `env.observation_space`, `env.action_space` — описание пространств состояний/действий

`terminated` — эпизод закончился естественным образом (например, дошли до цели или упали),
`truncated` — эпизод прерван искусственно (например, по лимиту шагов).


In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=False)
print("observation_space:", env.observation_space)
print("action_space:", env.action_space)

obs, info = env.reset(seed=0)
print("начальное состояние:", obs)

for step in range(5):
    action = env.action_space.sample()  # случайное действие
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"step={step} action={action} obs={obs} reward={reward} terminated={terminated}")
    if terminated or truncated:
        obs, info = env.reset()

env.close()


`FrozenLake` — классическая табличная MDP-среда: агент идёт по льду 4x4 из точки S в точку G,
избегая дыр H. С `is_slippery=True` переходы становятся стохастическими (лёд скользкий) —
пригодится на неделе 2 для Policy/Value Iteration.

## 2. Своя среда многорукого бандита

Напишем среду в стиле Gymnasium, но без наследования от `gym.Env` — минимальный класс,
которого достаточно для наших целей.


In [ ]:
class BernoulliBanditEnv:
    """K рук, у каждой руки k своя вероятность успеха p_k (награда 0 или 1)."""

    def __init__(self, probs, rng=None):
        self.probs = np.array(probs, dtype=float)
        self.n_arms = len(probs)
        self.rng = rng or np.random.default_rng()

    def pull(self, arm: int) -> float:
        return float(self.rng.random() < self.probs[arm])

    @property
    def optimal_mean(self) -> float:
        return self.probs.max()


bandit = BernoulliBanditEnv(probs=[0.1, 0.5, 0.3, 0.55, 0.45], rng=rng)
print("оптимальная рука:", bandit.probs.argmax(), "с вероятностью", bandit.optimal_mean)


## 3. Агенты

Реализуем трёх агентов с общим интерфейсом: `select_arm()` и `update(arm, reward)`.


In [ ]:
class EpsilonGreedyAgent:
    def __init__(self, n_arms, epsilon=0.1, rng=None):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.rng = rng or np.random.default_rng()

    def select_arm(self) -> int:
        if self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.n_arms))
        return int(np.argmax(self.Q))

    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]


class UCB1Agent:
    def __init__(self, n_arms, c=2.0):
        self.n_arms = n_arms
        self.c = c
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.t = 0

    def select_arm(self) -> int:
        self.t += 1
        # пока не попробовали каждую руку хотя бы раз - пробуем её
        untried = np.where(self.N == 0)[0]
        if len(untried) > 0:
            return int(untried[0])
        bonus = self.c * np.sqrt(np.log(self.t) / self.N)
        return int(np.argmax(self.Q + bonus))

    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]


class ThompsonSamplingAgent:
    """Beta-Bernoulli conjugate prior: Beta(1,1) -> Beta(1+successes, 1+failures)."""

    def __init__(self, n_arms, rng=None):
        self.n_arms = n_arms
        self.alpha = np.ones(n_arms)
        self.beta = np.ones(n_arms)
        self.rng = rng or np.random.default_rng()

    def select_arm(self) -> int:
        samples = self.rng.beta(self.alpha, self.beta)
        return int(np.argmax(samples))

    def update(self, arm: int, reward: float):
        self.alpha[arm] += reward
        self.beta[arm] += 1 - reward


## 4. Сравнение по regret

Запустим каждого агента на одном и том же бандите и посчитаем накопленный regret.


In [ ]:
def run_agent(agent_factory, bandit_probs, n_steps=2000, n_seeds=20):
    cum_regrets = np.zeros((n_seeds, n_steps))
    for seed in range(n_seeds):
        local_rng = np.random.default_rng(seed)
        bandit = BernoulliBanditEnv(bandit_probs, rng=local_rng)
        agent = agent_factory(local_rng)
        regret = np.zeros(n_steps)
        for t in range(n_steps):
            arm = agent.select_arm()
            reward = bandit.pull(arm)
            agent.update(arm, reward)
            regret[t] = bandit.optimal_mean - bandit.probs[arm]
        cum_regrets[seed] = np.cumsum(regret)
    return cum_regrets.mean(axis=0)


bandit_probs = [0.1, 0.5, 0.3, 0.55, 0.45]
n_arms = len(bandit_probs)

agents = {
    "epsilon-greedy (eps=0.1)": lambda rng: EpsilonGreedyAgent(n_arms, epsilon=0.1, rng=rng),
    "UCB1 (c=2)": lambda rng: UCB1Agent(n_arms, c=2.0),
    "Thompson Sampling": lambda rng: ThompsonSamplingAgent(n_arms, rng=rng),
}

plt.figure(figsize=(7, 5))
for name, factory in agents.items():
    cum_regret = run_agent(factory, bandit_probs)
    plt.plot(cum_regret, label=name)

plt.xlabel("шаг t")
plt.ylabel("средний накопленный regret")
plt.title("Сравнение стратегий на 5-руком Bernoulli-бандите")
plt.legend()
plt.show()


Обратите внимание на форму кривых: у ε-greedy regret растёт **линейно** даже после
нахождения лучшей руки (постоянный шанс ε продолжать исследовать), тогда как у UCB1 и
Thompson Sampling рост **замедляется** — они меньше exploration'ят руки, в которых уже уверены.

## Что дальше

На домашнем задании нужно будет:

* провести более полное сравнение (несколько конфигураций рук, разные ε/c)
* реализовать вывод уравнений Беллмана вручную для пары простых MDP

А на следующей неделе перейдём от бандитов (без состояний) к полноценным MDP и научимся
находить оптимальную политику, зная модель среды (Dynamic Programming).
